# Assignment 1: Dynamic Programming

---

## Task 1) Edit Distances

Implement the [Hamming](https://en.wikipedia.org/wiki/Hamming_distance) and [Levenshtein](https://en.wikipedia.org/wiki/Levenshtein_distance) (edit) distances and compute them for the for the following word pairs.
It may help to compute them by hand first.

<img src = "./assets/97090.jpg" width="33%" /> <img src = "./assets/97222.jpg" width="33%" /> <img src = "./assets/97669.jpg" width="33%" />

Aside from computing the distance (which is a scalar), do the backtrace and compute the edit transcript (and thus alignment).

---

In [2]:
import numpy as np
import math

WORD_PAIRS = [
    ("GCGTATGAGGCTAACGC", "GCTATGCGGCTATACGC"),
    ("kühler schrank", "schüler krank"),
    ("the longest", "longest day"),
    ("nicht ausgeloggt", "licht ausgenockt"),
    ("gurken schaben", "schurkengaben")
]

In [3]:
def hamming(s1: str, s2: str) -> int:
    """
    Compute the hamming distance between two strings.

    Arguments:
    s1: First string of word pair.
    s2: Second string of word pair.

    Returns the distance as int.
    """
    # Hint: Think about how you can deal with unequal word lengths.
    

    dist = abs(len(s1)-len(s2))
    for i in range(min(len(s1), len(s2))):
        if s1[i] != s2[i]:
            dist += 1 
            
    return dist
    
    ### END YOUR CODE

In [4]:
for wordpair in WORD_PAIRS:
    dist = hamming(s1=wordpair[0], s2=wordpair[1])
    print("hamming('{}', '{}') = {}".format(
        wordpair[0], wordpair[1], dist
    ))

### EXPECTED
# hamming('GCGTATGAGGCTAACGC', 'GCTATGCGGCTATACGC') = 10
# hamming('kühler schrank', 'schüler krank') = 13
# hamming('the longest', 'longest day') = 11
# hamming('nicht ausgeloggt', 'licht ausgenockt') = 4
# hamming('gurken schaben', 'schurkengaben') = 14

hamming('GCGTATGAGGCTAACGC', 'GCTATGCGGCTATACGC') = 10
hamming('kühler schrank', 'schüler krank') = 13
hamming('the longest', 'longest day') = 11
hamming('nicht ausgeloggt', 'licht ausgenockt') = 4
hamming('gurken schaben', 'schurkengaben') = 14


In [5]:
def levenshtein(s1: str, s2: str, cost={'m': 0, 's': 1, 'i': 1, 'd': 1}) -> (int, str):
    """
    Compute the levenshtein (edit) distance between two strings.

    Arguments:
    s1: First string of word pair.
    s2: Second string of word pair.

    Returns the distance as int and edit transcript as string.
    """
    # Hint: The probably most intuitive approach is bottom-up.
    
    ### YOUR CODE HERE

    D = np.zeros((len(s1)+1, len(s2)+1))
    D[0, 1:] = range(1, len(s2) + 1)
    D[1:, 0] = range(1, len(s1) + 1)
    OP = np.empty((len(s1)+1, len(s2)+1), dtype=str)
    for i in range(1, len(s1) +1):
        for j in range(1, len(s2)+1):
            if s1[i-1] == s2[j-1]:
                delta, delta_op = cost['m'], 'm'
            else:
                delta, delta_op = cost['s'], 's'
            val, op = min((D[i-1, j-1] + delta, delta_op),          # same or replacement
                          (D[i, j-1] + cost['i'], 'i'),        # insertion
                          (D[i-1, j] + cost['d'], 'd')         # deletion
                          )
            D[i, j] = val
            OP[i, j] = op
    #print(OP)

    op_string = ''
    y = len(s1)
    x = len(s2)
    while y > 1 or x > 1:   # we start at the last operation and backtrack our way to the first one
        #print(x, y)
        op_string = OP[y, x] + op_string    
        if OP[y, x] == 'd' or OP[y, x-1] == '':
            y -= 1
        elif OP[y, x] == 'i' or OP[y-1, x] == '':
            x -= 1
        elif OP[y, x] == 'm':
            x -= 1
            y -= 1
        elif OP[y, x] == 's':
            x -= 1
            y -= 1
    
    op_string = OP[y, x] + op_string   



    return int(D[len(s1), len(s2)]), op_string
    ### END YOUR CODE

In [6]:
for wordpair in WORD_PAIRS:
    dist, transcript = levenshtein(s1=wordpair[0], s2=wordpair[1])
    print("levenshtein('{}', '{}') = {} ({})".format(
        wordpair[0], wordpair[1], dist, transcript
    ))

### EXPECTED
# levenshtein('GCGTATGAGGCTAACGC', 'GCTATGCGGCTATACGC') = 3 (mmdmmmmsmmmmmimmmm)
# levenshtein('kühler schrank', 'schüler krank') = 6 (ssmimmmmsddmmmm)
# levenshtein('the longest', 'longest day') = 8 (ddddmmmmmmmiiii)
# levenshtein('nicht ausgeloggt', 'licht ausgenockt') = 4 (smmmmmmmmmmsmssm)
# levenshtein('gurken schaben', 'schurkengaben') = 7 (siimmmmmsdddmmmm)

levenshtein('GCGTATGAGGCTAACGC', 'GCTATGCGGCTATACGC') = 3 (mmdmmmmsmmmmmimmmm)
levenshtein('kühler schrank', 'schüler krank') = 6 (ssmimmmmsddmmmm)
levenshtein('the longest', 'longest day') = 8 (sdddmmmmmmmiiii)
levenshtein('nicht ausgeloggt', 'licht ausgenockt') = 4 (smmmmmmmmmmsmssm)
levenshtein('gurken schaben', 'schurkengaben') = 7 (siimmmmmsdddmmmm)


---

## Task 2) Basic Spelling Correction using Edit Distance

For spelling correction, we will use prior knowledge, to put _some_ learning into our system.

The underlying idea is the _Noisy Channel Model_, that is: The user _intends_ to write a word `w`, but through some noise in the process, happens to type the word `x`.

The correct word $\hat{w}$ is that word, that is a valid candidate and has the highest probability:

$$
\begin{eqnarray}
\DeclareMathOperator*{\argmax}{argmax}
\hat{w} & = & \argmax_{w \in V} P(w | x) \\
        & = & \argmax_{w \in V} \frac{P(x|w) P(w)}{P(x)} \\
        & = & \argmax_{w \in V} P(x|w) P(w)
\end{eqnarray}
$$

1. The candidates $V$ can be obtained from a _vocabulary_.
2. The probability $P(w)$ of a word $w$ can be _learned (counted) from data_.
3. The probability $P(x|w)$ is more complicated... It could be learned from data, but we could also use a _heuristic_ that relates to the edit distance, e.g. rank by distance.

You may use [Peter Norvig's count_1w.txt](http://norvig.com/ngrams/) file, [local mirror](res/count_1w.tar.bz2).
Note that it may help to restrict to the first 10k words to get started.

---

In [ ]:
EXAMPLES = [
    "pirates",    # in-voc
    "pirutes",    # pirates?
    "continoisly",  # continuously?
]

In [75]:
### TODO: Prepare the vocabulary

### YOUR CODE HERE

voc_dict = {} 
with open('norvig_wordCount.txt') as file:
    total_words = 0
    for line in file:
        word, count = line.split()
        voc_dict[word] = int(count)       # a dict is created with the shape {word: count}
        #print(word, count)
        total_words += int(count)
    print(total_words)
#print(voc_dict)

for key, val in voc_dict.items():
    voc_dict[key] = val/total_words
#print(voc_dict)

### END YOUR CODE

588124220187


In [ ]:
def suggest(w: str, dist_fn, max_cand=5) -> list:
    """
    Compute suggestions for spelling correction using edit distance.
    
    Arguments:
    w: Word in question.
    dist_fn: Distance function to use (e.g. levenshtein).
    max_cand: Maximum number of suggestions.

    Returns a list of tuples (word, dist, score) sorted by score and distance.
    """
    ### YOUR CODE HERE
    out = []
    if word in voc_dict:
        max_cand = 1
    
    for k, val in voc_dict.items():

        dist = dist_fn(w, k)[0]
        score = math.log(val)
        out.append((k, dist, score))
    out.sort(key=lambda l: l[1])
    #print(out)
    return out[:max_cand]
    
    ### END YOUR CODE

In [11]:
# How does your suggest function behave with levenshtein distance?

for word in EXAMPLES:
    suggestions = suggest(w=word, dist_fn=levenshtein, max_cand=3)
    print("{} {}".format(word, suggestions))

### EXPECTED
### Notice: your scores may vary!
# pirates [('pirates', 0, -11.408058827802126)]
# pirutes [('pirates', 1, -11.408058827802126), ('minutes', 2, -8.717825438953103), ('viruses', 2, -11.111468702571859)]
# continoisly [('continously', 1, -15.735337826575178), ('continuously', 2, -11.560071979871001), ('continuosly', 2, -17.009283000138204)]

pirates [('pirates', 0, -11.408058827802126)]
pirutes [('pirates', 1, -11.408058827802126), ('minutes', 2, -8.717825438953103), ('viruses', 2, -11.111468702571859)]
continoisly [('continously', 1, -15.735337826575178), ('continuously', 2, -11.560071979871001), ('continuosly', 2, -17.009283000138204)]


### Food for Thought

- How would you modify the implementation so that it works fast for large lexica (eg. the full file above)?
- How would you modify the implementation so that it works "while typing" instead of at the end of a word?
- How do you handle unknown/new words?

---

## Task 3) Needleman-Wunsch: Keyboard-aware Auto-Correct

In the parts 1 and 2 above, we applied uniform cost to all substitutions.
This does not really make sense if you look at a keyboard: the QWERTY layout will favor certain substitutions (eg. _a_ and _s_), while others are fairly unlikely (eg. _a_ and _k_).

Implement the [Needleman-Wunsch algorithm](https://en.wikipedia.org/wiki/Needleman–Wunsch_algorithm) which is very similar to the [Levenshtein distance](https://en.wikipedia.org/wiki/Levenshtein_distance), however it doesn't _minimize the cost_ but _maximizes the similarity_.
For a good measure of similarity, implement a metric that computes a meaningful weight for a given character substitution.

---

In [ ]:
def keyboardsim(s1: str, s2: str) -> float:
    ### YOUR CODE HERE
    kb_map = {
        'q': (0,0), 'w': (0,1), 'e': (0,2), 'r': (0,3), 't': (0,5),'z': (0,6), 'u': (0,7), 'i': (0,8), 'o': (0,9), 'p': (0,10),
        'a': (1,0), 's': (1,1), 'd': (1,2), 'f': (1,3), 'g': (1,5),'h': (1,6), 'j': (1,7), 'k': (1,8), 'l': (1,9),
        'y': (2,0), 'x': (2,1), 'c': (2,2), 'v': (2,3), 'b': (2,4),'n': (2,6), 'm': (2,7)
    }
    sim = - math.dist(kb_map[s1], kb_map[s2])
    #print(s1, s2, sim)
    return sim
    ### END YOUR CODE


def nw(s1: str, s2: str, d: float, sim_fn) -> float:
    """
    Apply needleman-wunsch algorithm.
    
    Arguments:
    s1: First string of word pair.
    s2: Second string of word pair.
    d: Gap penalty score.
    sim_fn: Similarity function to use.

    Returns the score as float.
    """
    ### YOUR CODE HERED = np.zeros((len(s1)+1, len(s2)+1))

    D = np.zeros((len(s1)+1, len(s2)+1))
    
    D[0, 1:] = range(-1, -(len(s2)+1), -1)
    D[1:, 0] = range(-1, -(len(s1)+1), -1)
    #print(D)
    for i in range(1, len(s1) +1):
        for j in range(1, len(s2)+1):
            #print(s1[i-1], s2[j-1])
            val = max((D[i-1, j-1] + keyboardsim(s1[i-1], s2[j-1])),          # same or replacement
                        (D[i, j-1] + d),        # insertion
                        (D[i-1, j] + d)         # deletion
                          )
            D[i, j] = val


    return int(D[len(s1), len(s2)])
    ### END YOUR CODE




def nw_based_dist(s1: str, s2: str) -> (int, str):
    """
    Compute the needleman-wunsch distance between two strings.
    
    Arguments:
    s1: First string of word pair.
    s2: Second string of word pair.
    
    Returns the distance as int and <unsupported> string.
    """
    ### YOUR CODE HERE
    gap_penalty = -4
    score = nw(s1, s2, gap_penalty, keyboardsim)
    return -score, "<unsupported>"
    # Hint: return dist, "<unsupported>"
    
    ### END YOUR CODE

In [96]:
# How does your suggest function behave with nw and a keyboard-aware similarity?

for word in EXAMPLES:
    suggestions = suggest(w=word, dist_fn=nw_based_dist, max_cand=3)
    print("{} {}".format(word, suggestions))

pirates [('pirates', 0.0, -11.408058827802126)]
pirutes [('porites', 2.0, -16.812645253397843), ('spirited', 3.0, -12.77810841818154), ('rites', 3.0, -12.956638697338697)]
continoisly [('continously', 1.0, -15.735337826575178), ('continuosly', 3.0, -17.009283000138204), ('continuously', 5.0, -11.560071979871001)]


### Food for Thought

- What could be better heuristics for similarity resp. cost of substitution than the one above?
- What about capitalization, punctiation and special characters?
- What about swipe-to-type?